# ClauseLens — Node 1 → Node 2 → Node 3 pipeline (Kaggle)

Runs PDF chunking (Node 1), indexing (Node 2) and clause alignment with the CUAD-fine-tuned Gemma (Node 3).
Needs a GPU and a Kaggle secret named `HF-Token`.

In [ ]:
%%capture
import os, re

# Must be set BEFORE torch initializes CUDA
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {
        '2.10': '0.0.34',
        '2.9': '0.0.33.post1',
        '2.8': '0.0.32.post2'
    }.get(v, "0.0.34")

    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"

!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install "huggingface_hub>=1.5.0,<2.0"
!pip install torchcodec
!pip install -q pymupdf numpy sentence-transformers faiss-cpu rank-bm25 networkx

import torch
torch._dynamo.config.recompile_limit = 64

## Setup

In [ ]:
import unsloth  # must be imported before transformers / peft
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF-Token")
print("HF token loaded: SUCCESS")

In [ ]:
import os
import sys

REPO_PATH = "/kaggle/working/ClauseLens"

if not os.path.exists(REPO_PATH):
    !git clone https://github.com/Rhivu10/ClauseLens.git /kaggle/working/ClauseLens
else:
    print("ClauseLens repository already exists — pulling latest.")
    !git -C /kaggle/working/ClauseLens pull

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

from nodes.node_1 import process_documents
from nodes.node_2 import prepare_chunks, Node2Indexer, semantic_search
from nodes.node_3 import Node3Config, load_gemma_with_adapter, GemmaVerifier, Node3Aligner, show
print("Node 1 / 2 / 3 import: SUCCESS")

## Paths and settings

In [ ]:
CHECKPOINT_PATH = "/kaggle/input/datasets/shrivyaazsharmajb/checkpoint/checkpoint-2091"

Doc1 = "/kaggle/input/datasets/shrivyaazsharmajb/cuad-datasets-1/AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.19_11788293_EX-10.19.pdf"
Doc2 = "/kaggle/input/datasets/shrivyaazsharmajb/cuad-datasets-1/ArconicRolledProductsCorp_20191217_10-12B_EX-2.7_11923804_EX-2.7.pdf"

CHUNK_MAX_CHARS = 1500   # Node 1
CHUNK_SPLIT_LEVEL = 2    # split on 1., 1.1, ARTICLE, SECTION

config = Node3Config()   # edit e.g. config.similarity_threshold after calibration

for p in (CHECKPOINT_PATH, Doc1, Doc2):
    print("exists:", os.path.exists(p), p)

## Load fine-tuned Gemma

In [ ]:
model, tokenizer = load_gemma_with_adapter(
    CHECKPOINT_PATH,
    max_seq_length=config.max_seq_length,
    hf_token=HF_TOKEN,
)
print("Gemma + CUAD LoRA adapter: READY")
print(f"Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

## Node 1 — document processing

In [ ]:
import numpy as np

result = process_documents(
    Doc1, Doc2,
    start_new_chunk_on_level=CHUNK_SPLIT_LEVEL,
    max_chars=CHUNK_MAX_CHARS,
)

# Quality check — look at this before trusting Node 3
for doc_id, d in result.items():
    lens = [len(c["text"]) for c in d["chunks"]]
    print(doc_id, "lines:", d["element_count"], "| chunks:", len(lens),
          "| len p10/p50/p90/max:", np.percentile(lens, [10, 50, 90, 100]).astype(int).tolist(),
          "| tiny(<80):", sum(l < 80 for l in lens),
          "| over max:", sum(l > CHUNK_MAX_CHARS for l in lens))

## Node 2 — indexing

In [ ]:
all_chunks = prepare_chunks(
    result,
    version_ids={"document_a": "version_a", "document_b": "version_b"},
)
indexer = Node2Indexer()
index_bundle = indexer.build(all_chunks)

print("Total chunks:", len(all_chunks))
print("FAISS vectors:", index_bundle.vector_index.ntotal)
print("Graph nodes / edges:", index_bundle.knowledge_graph.number_of_nodes(),
      index_bundle.knowledge_graph.number_of_edges())

# Sanity check: scores must be descending cosine similarities for the gate to be valid
_scores = [x["score"] for x in semantic_search("indemnification", index_bundle, indexer.embedding_model, top_k=5)]
print("scores:", [round(s, 3) for s in _scores])
if _scores != sorted(_scores, reverse=True) or max(_scores) > 1.0001:
    print("WARNING: scores are not descending cosine similarities — check Node 2.")
else:
    print("OK: scores look like descending cosine similarities.")

## Node 3 — clause alignment

In [ ]:
verifier = GemmaVerifier(model, tokenizer, config)
aligner = Node3Aligner(all_chunks, index_bundle, indexer.embedding_model, verifier)
print("Node 3 ready | threshold:", config.similarity_threshold,
      "| candidate_k:", config.candidate_k, "| chars/side:", config.text_chars)

### Threshold calibration

In [ ]:
rows = aligner.best_scores("document_a", "document_b")
vals = np.array([s for _, s in rows])
print("chunks:", len(vals))
print("score percentiles p0/p10/p25/p50/p75/p90/p100:",
      np.round(np.percentile(vals, [0, 10, 25, 50, 75, 90, 100]), 3).tolist())
print(f"gate {config.similarity_threshold}: {int((vals >= config.similarity_threshold).sum())} of {len(vals)} chunks pass")
print("Lowest 5:", [(cid, round(s, 3)) for cid, s in sorted(rows, key=lambda x: x[1])[:5]])
print("Highest 5:", [(cid, round(s, 3)) for cid, s in sorted(rows, key=lambda x: -x[1])[:5]])

### Tests

In [ ]:
def pick_chunk(doc_id, keywords, fallback_index=0):
    hits = [c for c in all_chunks
            if c["document_id"] == doc_id and any(k in c["text"].lower() for k in keywords)]
    if hits:
        return hits[0], True
    return [c for c in all_chunks if c["document_id"] == doc_id][fallback_index], False

# TEST 0 — self-match guard
_sc = next(c for c in all_chunks if c["document_id"] == "document_b")
try:
    aligner.align_source_clause(_sc, "document_b")
    print("TEST 0 FAILED: self-match was not rejected")
except ValueError as e:
    print("TEST 0 PASSED:", e)

In [ ]:
for title, keywords in [
    ("TEST 1 — INDEMNIFICATION", ["indemnif"]),
    ("TEST 2 — GOVERNING LAW", ["governing law", "governed by", "laws of the state"]),
]:
    src, found = pick_chunk("document_a", keywords)
    print("=" * 80)
    print(title + ("" if found else "  (keyword not found; using first chunk)"))
    print("=" * 80)
    show(aligner.align_source_clause(src, "document_b"), src)
    print()

In [ ]:
# TEST 3 — unmatched clause (lowest best-score chunk in document_a)
src3 = aligner.chunk_by_id[min(rows, key=lambda x: x[1])[0]]
show(aligner.align_source_clause(src3, "document_b"), src3)
print()
print("Expected: status BELOW_THRESHOLD (no Gemma call). If COMPARED, raise")
print("config.similarity_threshold using the calibration numbers above.")

### Optional: full document run

In [ ]:
RUN_FULL = False           # set True to align every document_a chunk against document_b
MAX_SOURCE_CHUNKS = None   # e.g. 10 for a smoke test

if RUN_FULL:
    report = aligner.align_documents("document_a", "document_b", max_source_chunks=MAX_SOURCE_CHUNKS)
    print("counts:", report["counts"])
    print("REMOVED (in A, not in B):", report["removed"])
    print("ADDED   (in B, not in A):", report["added"])

print("Allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("Reserved :", round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")